# Transformer Architecture: Complete Technical Guide

## Table of Contents
1. [Self-Attention Mechanism](#1-self-attention-mechanism) — Intuition, Q/K/V, scaled dot-product, step-by-step
2. [All Attention Variants](#2-all-attention-variants) — MHA, Masked, Cross, MQA, GQA, MLA, Flash, Sparse, Ring, Differential
3. [Positional Encoding](#3-positional-encoding) — Sinusoidal, Learned, RoPE, ALiBi
4. [Encoder Architecture](#4-encoder-architecture) — Residual connections, LayerNorm, FFN, stacking
5. [Decoder Architecture](#5-decoder-architecture) — Masked attention, cross-attention, teacher forcing
6. [Training & Optimization](#6-training--optimization) — Loss, LR schedule, mixed precision, parallelism
7. [Modern Architectures & Interview Prep](#7-modern-architectures--interview-prep) — BERT/GPT/T5, KV-cache, speculative decoding, Q&A

---

# 1. Self-Attention Mechanism

## 1.1 Why Self-Attention? The Core Motivation

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>The fundamental question:</b> Given a sequence of words, how should each word <i>look at</i> every other word to build a rich, context-aware representation?
</div>

### The Problem with Previous Approaches

| Architecture | How it sees context | Limitation |
|---|---|---|
| **RNN/LSTM** | Sequential: token-by-token, left-to-right | Long-range dependencies decay over steps; cannot parallelize (each step depends on previous) |
| **CNN** | Local: fixed-size sliding windows | Limited receptive field per layer; needs many stacked layers for distant tokens to interact |
| **Self-Attention** | **Global: every token attends to every other token in ONE step** | O(n²) complexity, but fully parallelizable and captures any-distance dependencies instantly |

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Key insight:</b> Self-attention computes a <b>weighted average of all positions</b> for each position. The weights are <b>learned dynamically</b> based on the content — not fixed by architecture. This means the model can learn to focus on whatever is relevant, regardless of distance.
</div>

---

## 1.2 Mathematical Foundation

**The Attention Formula (Scaled Dot-Product Attention):**

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

| Symbol | Name | Intuition |
|---|---|---|
| **Q** (Query) | "What am I looking for?" | Each token broadcasts what information it needs |
| **K** (Key) | "What do I contain?" | Each token advertises what information it has |
| **V** (Value) | "Here is my actual information" | The content that gets aggregated |
| **d_k** | Key dimension | Scaling factor to prevent softmax saturation |

The self-attention mechanism is also called **scaled dot-product attention**.

![Attention formula diagram](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/2fecf5e4-32ba-4687-b9af-18df94aa695d.png)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Example: Self-Attention Implementation
class SelfAttention(nn.Module):
    def __init__(self, d_model, d_k):
        super(SelfAttention, self).__init__()
        self.d_model = d_model
        self.d_k = d_k
        
        # Linear transformations for Q, K, V
        self.W_q = nn.Linear(d_model, d_k, bias=False)
        self.W_k = nn.Linear(d_model, d_k, bias=False)
        self.W_v = nn.Linear(d_model, d_k, bias=False)
        
    def forward(self, x):
        batch_size, seq_len, d_model = x.size()
        
        # Compute Q, K, V
        Q = self.W_q(x)  # [batch_size, seq_len, d_k]
        K = self.W_k(x)  # [batch_size, seq_len, d_k]
        V = self.W_v(x)  # [batch_size, seq_len, d_k]
        
        # Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        # Apply softmax
        attention_weights = F.softmax(scores, dim=-1)
        
        # Compute weighted sum
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights

# Example usage
d_model = 12
d_k = 12
seq_len = 3

# Create sample input (word embeddings for "I am good")
x = torch.randn(1, seq_len, d_model)  # batch_size=1, seq_len=3, d_model=12

# Initialize self-attention
self_attention = SelfAttention(d_model, d_k)

# Forward pass
output, attention_weights = self_attention(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", attention_weights.shape)
print("\nAttention weights:")
print(attention_weights.squeeze().detach().numpy())


In [ ]:
# Visualize Attention Weights
def visualize_attention(attention_weights, words=["I", "am", "good"]):
    """Visualize attention weights as a heatmap"""
    plt.figure(figsize=(8, 6))
    
    # Convert to numpy and remove batch dimension
    attn = attention_weights.squeeze().detach().numpy()
    
    # Create heatmap
    sns.heatmap(attn, 
                xticklabels=words, 
                yticklabels=words,
                annot=True, 
                fmt='.3f',
                cmap='Blues',
                cbar_kws={'label': 'Attention Weight'})
    
    plt.title('Self-Attention Weights')
    plt.xlabel('Key (attended to)')
    plt.ylabel('Query (attending from)')
    plt.tight_layout()
    plt.show()

# Visualize the attention weights
visualize_attention(attention_weights)


## 1.3 Step-by-Step Walkthrough: "I am good"

### Step 0: Input Embeddings → Input Matrix

Each word is converted to a dense vector (embedding). The input sentence "I am good" becomes an **input matrix X** of shape `[sentence_length × embedding_dimension]` = `[3 × 12]`:

![Input matrix](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/c3132147-b254-4212-bf73-d5c7d3e9c3fa.png)

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
Embeddings are learned during training — they are the vector representation of each word capturing semantic meaning.
</div>

### Step 0.5: Create Q, K, V via Learned Projections

From the input matrix **X**, we create three matrices using learned weight matrices:

$$Q = X \cdot W_Q, \quad K = X \cdot W_K, \quad V = X \cdot W_V$$

Weight matrices $W_Q, W_K, W_V$ are **randomly initialized** and their optimal values are **learned during training**.

![Q, K, V creation](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/816dafcc-dc33-4166-a0ce-76d8dfba5786.png)

- Row 1 ($q_1, k_1, v_1$) → query, key, value vectors for **"I"**
- Row 2 ($q_2, k_2, v_2$) → query, key, value vectors for **"am"**
- Row 3 ($q_3, k_3, v_3$) → query, key, value vectors for **"good"**

---

### Step 1: Compute Similarity Scores ($QK^T$)

Compute the **dot product** between the query matrix and the transposed key matrix. The dot product tells us **how similar each pair of words is** — it produces a similarity score for every word pair:

![QK^T dot product](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/f0ca2ff8-f9bd-4e26-9b39-1f0b66973d8b.png)

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
Each cell (i, j) in this matrix represents: "How much should word <i>i</i> attend to word <i>j</i>?"
</div>

---

### Step 2: Scale by $\sqrt{d_k}$

Divide all scores by $\sqrt{d_k}$ (square root of the key vector dimension):

![Scaling](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/e15fbb4d-5083-476f-b27f-76d299961545.png)

<div style="background-color: #fcf8e3; border: 1px solid #faebcc; color: #8a6d3b; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Why scale?</b> As $d_k$ grows, dot products grow in magnitude, pushing softmax into regions where gradients are extremely small (saturation). Dividing by $\sqrt{d_k}$ keeps the variance of scores ≈ 1, ensuring stable gradients and meaningful softmax outputs.
</div>

---

### Step 3: Apply Softmax → Score Matrix

Apply softmax row-wise to normalize scores into probabilities (range 0 to 1, each row sums to 1):

![Softmax scores](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/6ef2abf9-c9ab-4b33-9dd4-0031cee677a0.png)

Now we can read off **how much each word attends to every other word**. Each row is an attention distribution for one word.

---

### Step 4: Compute Attention Matrix (Weighted Sum with V)

Multiply the score matrix by the value matrix **V**. Each output row is a **weighted combination of all value vectors**, where the weights come from the attention scores:

![Attention matrix](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/5f3d7bd4-2854-4cd9-a875-c694d8bd52f0.png)

**Example — Self-attention of the word "I":**

![Self-attention of "I"](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/86e1dbc8-dc75-4f28-9683-d70b20bf22e6.png)

> The self-attention output for "I" contains **90%** of v₁ (I), **7%** of v₂ (am), and **3%** of v₃ (good). This means the representation of "I" is mostly itself but enriched with context from the other words.

---

### 1.4 Coreference Resolution: Why Self-Attention Matters

Consider: *"The **dog** didn't cross the street because **it** was too tired."*

What does "it" refer to? A human knows it means "dog", not "street". Self-attention learns this:

![Coreference example](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/e5c0e9a1-7930-4ddc-aa28-992822049183.png)

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
The self-attention value of "it" contains <b>~100% of the value vector for "dog"</b>. The model has learned that "it" refers to "dog" and not "food" or "street". This is the power of self-attention — it dynamically discovers relationships between any pair of words, regardless of distance.
</div>

![Attention is all you need](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/8bf9914c-081f-409f-91da-9c4fc8af0aa3.png)



## Multi-head attention mechanism
<div style="background-color: #fcf8e3; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
For better representation to ambiguous word and remove the dominance affect of specific word on other words in sentences .
</div>

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">

The multi-head attention layer is an extension of the self-attention layer. It splits the queries, keys, and values into multiple heads, each with a smaller dimension, and performs self-attention on each head separately. Then it concatenates the outputs of all heads and applies another linear projection. <b>This allows the model to attend to different aspects or features of the input sequence simultaneously.
    </b></div>

In order to maintain correct representation  `instead of computing a single attention matrix, we will compute multiple attention matrices and then concatenate their results`. 

> The idea behind using multi-head attention is that instead of using a single attention head, if we use multiple attention heads, then our attention matrix will be more accurate.

Suppose we have eight attention matrices, 21 to z8  to ; then, we can just concatenate all the attention heads (attention matrices) and multiply the result by a new weight matrix, Wo , and create the final attention matrix as shown: 

![](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/15ea0152-cd90-4ffd-832a-09e5b931e185.png)



---

# 3. Positional Encoding

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Core Problem:</b> Unlike RNNs/LSTMs that process tokens sequentially (and thus inherently know token order), Transformers process all tokens <b>in parallel</b>. Without positional encoding, the sentence "dog bites man" and "man bites dog" would produce <b>identical</b> attention outputs -- the model has no concept of word order.
</div>

## 3.1 Sinusoidal Positional Encoding (Original "Attention Is All You Need")

The positional encoding matrix has the **same dimension as the input embedding matrix** so they can be summed element-wise:

$$\text{Input to Encoder} = \text{WordEmbedding}(x) + \text{PositionalEncoding}(pos)$$

![PE formula](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/ae0bba23-b709-4214-ab1e-37bef2fedb80.png)

**The sinusoidal formulas:**

$$PE_{(pos, 2i)} = \sin\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right) \qquad PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

- `pos` = position of the token in the sequence (0, 1, 2, ...)
- `i` = dimension index (0, 1, 2, ..., d_model/2 - 1)
- `d_model` = embedding dimension

![Sin/cos computation](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/21aa0af2-6bb1-452a-8de7-2db89751d434.png)

### Why sin and cos at different frequencies?

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Key Intuition:</b>
<ol>
<li><b>Unique encoding per position:</b> Each position gets a unique pattern across dimensions -- like a "fingerprint".</li>
<li><b>Relative positions via linear transformation:</b> For any fixed offset <code>k</code>, <code>PE(pos+k)</code> can be expressed as a linear function of <code>PE(pos)</code>. This lets the model learn to attend to relative positions (e.g., "the word 3 positions back").</li>
<li><b>Bounded values:</b> sin/cos are always in [-1, 1], so they do not distort the magnitude of embeddings.</li>
<li><b>Multi-scale patterns:</b> Low dimensions oscillate slowly (capture long-range position), high dimensions oscillate fast (capture fine-grained position). Think of it like a binary clock -- each "bit" flips at a different rate.</li>
</ol>
</div>

![PE values](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/e7919233-30fe-4f57-b640-0de5b6f3ea48.png)

![PE matrix](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/96739439-541e-45b1-95c0-2052fd761024.png)

The final positional encoding matrix **P**:

![Final PE](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/d0600945-874a-400f-8a2c-7c54c9eec48a.png)

---

## 3.2 Learned vs Fixed Positional Encodings

| Aspect | Fixed (Sinusoidal) | Learned |
|---|---|---|
| **How** | Deterministic sin/cos functions | Embedding lookup table trained with the model |
| **Extrapolation** | Theoretically generalizes to unseen lengths (but limited in practice) | Cannot extrapolate beyond max training length |
| **Parameters** | Zero extra parameters | `max_len x d_model` extra parameters |
| **Performance** | Comparable to learned for standard lengths | Slightly better on tasks within training length |
| **Used by** | Original Transformer, some T5 variants | BERT, GPT-2 |

<div style="background-color: #fcf8e3; border: 1px solid #faebcc; color: #8a6d3b; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Interview Note:</b> The original paper showed sinusoidal and learned encodings perform nearly identically. The real breakthroughs in positional encoding came later with <b>RoPE</b> and <b>ALiBi</b>.
</div>

---

## 3.3 RoPE -- Rotary Position Embedding

Used in: **LLaMA, Mistral, Qwen, PaLM, GPT-NeoX, CodeLlama**

**Core idea:** Instead of *adding* position information to embeddings, RoPE *rotates* the query and key vectors by an angle proportional to their position.

**How it works:**
1. Group consecutive pairs of dimensions: $(x_0, x_1), (x_2, x_3), \ldots$
2. For position `pos` and dimension pair `i`, apply a 2D rotation by angle $\theta_i = pos \cdot \alpha^{-2i/d}$:

$$\begin{pmatrix} x'_{2i} \\ x'_{2i+1} \end{pmatrix} = \begin{pmatrix} \cos\theta_i & -\sin\theta_i \\ \sin\theta_i & \cos\theta_i \end{pmatrix} \begin{pmatrix} x_{2i} \\ x_{2i+1} \end{pmatrix}$$

3. The dot product $q_m \cdot k_n$ naturally becomes a function of the **relative position** $(m - n)$, because rotation angles subtract: $\theta_m - \theta_n$.

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Why RoPE is better:</b>
<ul>
<li><b>Relative position naturally encoded</b> -- no need for separate relative position bias</li>
<li><b>Decays with distance</b> -- dot product between distant tokens naturally decreases, acting as a soft distance penalty</li>
<li><b>Length extrapolation</b> -- with techniques like NTK-aware scaling or YaRN, RoPE models can extrapolate far beyond training context length</li>
<li><b>No extra parameters</b> -- purely functional, like sinusoidal encoding</li>
</ul>
</div>

---

## 3.4 ALiBi -- Attention with Linear Biases

Used in: **BLOOM, MPT**

**Core idea:** Do not add any positional encoding to embeddings at all. Instead, add a **linear bias** directly to the attention scores based on the distance between query and key positions:

$$\text{Attention}_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}} - m \cdot |i - j|$$

Where `m` is a head-specific slope (different per attention head, set as geometric sequence, not learned).

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Why ALiBi works well:</b>
<ul>
<li><b>Excellent length extrapolation</b> -- trained on 1K tokens, can generalize to 10K+ tokens with minimal quality loss</li>
<li><b>Simplicity</b> -- just a subtraction on attention scores, no modification to Q/K</li>
<li><b>No extra parameters or computation</b></li>
<li><b>Different heads use different slopes</b> -- some heads focus locally, others globally</li>
</ul>
</div>

### Comparison Summary

| Method | Modifies | Relative Position | Length Extrapolation | Extra Params |
|---|---|---|---|---|
| Sinusoidal | Embeddings (add) | Implicit (linear transform) | Limited | None |
| Learned | Embeddings (add) | No (absolute only) | None | max_len x d |
| RoPE | Q, K (rotate) | Yes (natural) | Good (with scaling) | None |
| ALiBi | Attention scores (subtract) | Yes (linear bias) | Excellent | None |

---

# 4. Encoder Architecture

## 4.1 Encoder Block Overview

Each encoder block has **two sublayers**, each wrapped with a residual connection and layer normalization:

```
Input --> Multi-Head Self-Attention --> Add & Norm --> Feed-Forward Network --> Add & Norm --> Output
```

![Encoder block](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/e11862b0-33b2-4ff6-8ca8-e4bba4777beb.png)

---

## 4.2 Residual Connections (Skip Connections)

$$\text{SubLayerOutput} = \text{LayerNorm}(x + \text{SubLayer}(x))$$

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Why residual connections are critical:</b>
<ul>
<li><b>Gradient flow:</b> In deep networks (6-96 layers), gradients must flow back through every layer. Skip connections create a "gradient highway" that prevents vanishing gradients.</li>
<li><b>Identity mapping:</b> Each sublayer only needs to learn the <i>residual</i> (the difference from identity), which is easier to optimize.</li>
<li><b>Deep training:</b> Without residual connections, training transformers deeper than ~3 layers becomes extremely difficult.</li>
</ul>
</div>

---

## 4.3 Layer Normalization

**Formula:**

$$\text{LayerNorm}(x) = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

Where $\mu$ and $\sigma^2$ are the mean and variance computed **across the feature dimension** for each token independently. $\gamma$ and $\beta$ are learnable scale and shift parameters.

### Pre-LN vs Post-LN

| Variant | Formula | Used By | Pros |
|---|---|---|---|
| **Post-LN** (original) | `LN(x + SubLayer(x))` | Original Transformer, BERT | Better final performance with careful tuning |
| **Pre-LN** (modern) | `x + SubLayer(LN(x))` | GPT-2, GPT-3, LLaMA, most modern LLMs | Much more stable training, no warmup needed, easier to scale |

<div style="background-color: #fcf8e3; border: 1px solid #faebcc; color: #8a6d3b; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Why Layer Norm and not Batch Norm?</b>
<ul>
<li><b>Batch Norm</b> normalizes across the batch dimension -- it computes statistics over all samples in a mini-batch for each feature. This creates dependencies between samples and fails with variable-length sequences and small batches.</li>
<li><b>Layer Norm</b> normalizes across the feature dimension for each sample independently -- no dependency on batch size, works perfectly with variable-length sequences and autoregressive generation (batch size = 1 at inference).</li>
</ul>
</div>

### RMSNorm (Modern Alternative)

Used in **LLaMA, Mistral, Gemma**. Removes the mean-centering step for efficiency:

$$\text{RMSNorm}(x) = \gamma \odot \frac{x}{\sqrt{\frac{1}{d}\sum_{i=1}^{d} x_i^2 + \epsilon}}$$

~10-15% faster than LayerNorm with comparable performance.

---

## 4.4 Feed-Forward Network (FFN)

$$\text{FFN}(x) = W_2 \cdot \text{Activation}(W_1 x + b_1) + b_2$$

- $W_1$: projects from `d_model` to `d_ff` (expansion, typically `d_ff = 4 * d_model`)
- Activation: **ReLU** (original) or **GELU/SwiGLU** (modern)
- $W_2$: projects back from `d_ff` to `d_model` (compression)

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Important details:</b>
<ul>
<li>The FFN is applied <b>position-wise</b> -- the same network is applied independently to each token position.</li>
<li>FFN parameters are <b>shared across positions</b> within a layer but <b>different across layers</b>.</li>
<li>The FFN accounts for roughly <b>2/3 of total model parameters</b> (two large weight matrices per layer).</li>
<li><b>SwiGLU</b> (used in LLaMA/PaLM): <code>SwiGLU(x) = (xW_1) * SiLU(xW_gate)</code> -- uses a gating mechanism, requires 3 weight matrices but improves quality.</li>
</ul>
</div>

![Add and Norm](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/6fb3dfd2-7e3f-43fd-baac-0a7e846a41d9.png)

---

## 4.5 Stacking Encoders

The original Transformer uses **N=6 encoder layers** stacked on top of each other:

1. Input embeddings + positional encoding are fed to Encoder Layer 1
2. Output of Encoder Layer $i$ becomes the input to Encoder Layer $i+1$
3. The output of the **final encoder layer** is the encoder representation $R$ -- sent to every decoder layer

![Full encoder stack](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/9129d96e-28a6-451d-9a47-a5eb88559cd1.png)

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>What each layer "learns":</b> Lower layers tend to capture syntax and local patterns. Higher layers capture more abstract semantic relationships. This is analogous to CNNs learning edges -> textures -> objects.
</div>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

# ============================================================
# Complete Encoder Block Implementation
# ============================================================

class MultiHeadAttention(nn.Module):
    """Multi-Head Attention with optional masking."""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        
        # Project and reshape: [B, seq, d_model] -> [B, heads, seq, d_k]
        Q = self.W_q(query).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Scaled dot-product attention
        scores = Q @ K.transpose(-2, -1) / np.sqrt(self.d_k)  # [B, heads, seq_q, seq_k]
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.attn_dropout(attn_weights)
        
        # Weighted sum of values
        context = attn_weights @ V  # [B, heads, seq_q, d_k]
        
        # Concatenate heads: [B, seq_q, d_model]
        context = context.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        output = self.W_o(context)
        
        return output, attn_weights


class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network."""
    def __init__(self, d_model, d_ff, dropout=0.1, activation='relu'):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU() if activation == 'gelu' else nn.ReLU()
    
    def forward(self, x):
        return self.linear2(self.dropout(self.activation(self.linear1(x))))


class EncoderBlock(nn.Module):
    """Single Transformer Encoder Block (Post-LN variant)."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        # Sublayer 1: Multi-Head Self-Attention + Add & Norm
        attn_out, attn_weights = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_out))
        
        # Sublayer 2: FFN + Add & Norm
        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))
        
        return x, attn_weights


class TransformerEncoder(nn.Module):
    """Stack of N Encoder Blocks."""
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout) 
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)  # Final layer norm
    
    def forward(self, x, mask=None):
        all_attn_weights = []
        for layer in self.layers:
            x, attn_w = layer(x, mask)
            all_attn_weights.append(attn_w)
        return self.norm(x), all_attn_weights


# === Demo ===
d_model, num_heads, d_ff, num_layers = 128, 8, 512, 6
encoder = TransformerEncoder(num_layers, d_model, num_heads, d_ff)

x = torch.randn(2, 10, d_model)  # batch=2, seq_len=10
enc_output, attn_weights_list = encoder(x)

print(f"Encoder input shape:  {x.shape}")
print(f"Encoder output shape: {enc_output.shape}")
print(f"Number of layers:     {len(attn_weights_list)}")
print(f"Attn weights shape (per layer): {attn_weights_list[0].shape}")

# Count parameters
total_params = sum(p.numel() for p in encoder.parameters())
print(f"\nTotal encoder parameters: {total_params:,}")
print(f"  Per-layer breakdown:")
layer0 = encoder.layers[0]
attn_params = sum(p.numel() for p in layer0.self_attn.parameters())
ffn_params = sum(p.numel() for p in layer0.ffn.parameters())
norm_params = sum(p.numel() for p in layer0.norm1.parameters()) + sum(p.numel() for p in layer0.norm2.parameters())
print(f"    Attention: {attn_params:,} ({attn_params/total_params*100:.1f}%)")
print(f"    FFN:       {ffn_params:,} ({ffn_params/total_params*100:.1f}%)")
print(f"    LayerNorm: {norm_params:,} ({norm_params/total_params*100:.1f}%)")

---

# 5. Decoder Architecture

## 5.1 Decoder Block Overview

Each decoder block has **three sublayers** (vs two in the encoder), each with residual + layer norm:

```
Input --> Masked Multi-Head Self-Attention --> Add & Norm
      --> Cross-Attention (Encoder-Decoder) --> Add & Norm
      --> Feed-Forward Network              --> Add & Norm --> Output
```

![Decoder block](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/7dfda3b3-c4fa-4428-90f2-77f8bc86f33a.png)

---

## 5.2 Masked Multi-Head Self-Attention

### Why masking?

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>The autoregressive constraint:</b> During generation, the decoder predicts one token at a time. When predicting token at position <code>t</code>, it must NOT see tokens at positions <code>t+1, t+2, ...</code> -- that would be "cheating" (looking at the future). The causal mask enforces this by preventing attention to future positions.
</div>

### The -inf masking trick

Before applying softmax to the attention scores, we set all "future" positions to $-\infty$:

$$\text{score}_{ij} = \begin{cases} \frac{q_i \cdot k_j}{\sqrt{d_k}} & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

Since $\text{softmax}(-\infty) = 0$, future tokens get **zero attention weight** -- they are completely invisible.

![Masking](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/0c5b3654-a936-4d1b-95f3-0ae791ae57b7.png)

**Example:** To predict the word after `<sos>`, the model can only see `<sos>` itself. All words to its right are masked with $-\infty$:

![Mask values](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/c529bd7f-d3a0-44a2-97af-4ef32c97cd84.png)

```python
# Creating a causal mask
def causal_mask(seq_len):
    mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()
    return ~mask  # True where attention is allowed
```

---

## 5.3 Cross-Attention (Encoder-Decoder Attention)

This is the critical bridge between encoder and decoder. It receives **two inputs**:

1. **From the previous sublayer** (masked self-attention output): used to create **Q** (queries)
2. **From the encoder output** $R$: used to create **K** (keys) and **V** (values)

![Cross-attention inputs](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/07834a62-995a-48f7-bd8d-29db23d33910.png)

$$Q = M \cdot W_Q \quad(\text{from decoder}) \qquad K = R \cdot W_K, \quad V = R \cdot W_V \quad(\text{from encoder})$$

![Cross-attention Q,K,V](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/e9c312cf-811f-45ba-a407-31be1f6acf74.png)

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>What cross-attention computes:</b> For each target token (query), it asks "which source tokens should I attend to?" This is how the decoder "looks back" at the input. For translation, this is where "Je" learns to attend to "I", "suis" attends to "am", etc.
</div>

![Cross-attention meaning](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/c8d934e3-bad8-43ed-bc7a-0f134cb6e6bf.png)

**Dimension note:** Q has shape `[batch, tgt_len, d_model]`, K and V have shape `[batch, src_len, d_model]`. The attention matrix is `[batch, heads, tgt_len, src_len]` -- rectangular, not square!

---

## 5.4 Teacher Forcing (Training) vs Autoregressive Generation (Inference)

### During Training: Teacher Forcing

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
The decoder receives the <b>entire</b> target sequence (shifted right by 1) as input, with a causal mask to prevent looking ahead. All positions are predicted <b>in parallel</b> in a single forward pass.
<br><br>
<b>Input:</b> <code>[&lt;sos&gt;, Je, suis, bon]</code><br>
<b>Target:</b> <code>[Je, suis, bon, &lt;eos&gt;]</code><br>
<b>Why "teacher forcing":</b> The model always sees the <i>correct</i> previous tokens (from the ground truth), not its own predictions. This stabilizes training but can cause <b>exposure bias</b> (train/test mismatch).
</div>

### During Inference: Autoregressive Generation

```
Step 0: Input = [<sos>]                    --> Predict "Je"
Step 1: Input = [<sos>, Je]                --> Predict "suis"
Step 2: Input = [<sos>, Je, suis]          --> Predict "bon"
Step 3: Input = [<sos>, Je, suis, bon]     --> Predict <eos> (STOP)
```

Each step requires a **full forward pass** through the decoder. This is why decoder inference is slow and why optimizations like **KV-cache** are critical (covered in Section 7).

---

## 5.5 Linear + Softmax Output Layer

The decoder output (a vector of dimension `d_model` per position) is projected to vocabulary size and then softmaxed:

$$P(\text{next token}) = \text{softmax}(W_{\text{vocab}} \cdot h_{\text{decoder}} + b)$$

![Linear + softmax](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/8aeae8dd-beed-436e-80db-14c5dda14410.png)

<div style="background-color: #fcf8e3; border: 1px solid #faebcc; color: #8a6d3b; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Weight tying:</b> In many modern models, the output projection matrix $W_{\text{vocab}}$ is <b>tied</b> (shared) with the input embedding matrix. This reduces parameters and often improves performance since both matrices map between token IDs and d_model-dimensional space.
</div>

---

## 5.6 Full Decoder Flow Summary

1. Target tokens are embedded and positional encoding is added
2. **Masked self-attention:** Each target token attends only to itself and previous target tokens
3. **Cross-attention:** Each target token attends to all source tokens (encoder output)
4. **FFN:** Position-wise transformation
5. Steps 2-4 repeat for N decoder layers
6. **Linear + Softmax:** Final layer projects to vocabulary and picks the most probable token

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# Full Transformer Model: Encoder + Decoder
# ============================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model, self.num_heads = d_model, num_heads
        self.d_k = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.W_q(query).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        scores = Q @ K.transpose(-2, -1) / np.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = self.dropout(F.softmax(scores, dim=-1))
        out = (attn @ V).transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.W_o(out), attn


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(),
            nn.Dropout(dropout), nn.Linear(d_ff, d_model)
        )
    def forward(self, x):
        return self.net(x)


class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)
    
    def forward(self, x, src_mask=None):
        attn_out, attn_w = self.self_attn(x, x, x, src_mask)
        x = self.norm1(x + self.drop1(attn_out))
        x = self.norm2(x + self.drop2(self.ffn(x)))
        return x, attn_w


class DecoderLayer(nn.Module):
    """Single Decoder Layer with 3 sublayers."""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        # Sublayer 1: Masked self-attention
        self.masked_self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        # Sublayer 2: Cross-attention (encoder-decoder)
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        # Sublayer 3: FFN
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.drop1 = nn.Dropout(dropout)
        self.drop2 = nn.Dropout(dropout)
        self.drop3 = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, tgt_mask=None, src_mask=None):
        # 1. Masked self-attention (Q=K=V from decoder input)
        self_attn_out, self_attn_w = self.masked_self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.drop1(self_attn_out))
        
        # 2. Cross-attention (Q from decoder, K/V from encoder)
        cross_attn_out, cross_attn_w = self.cross_attn(x, encoder_output, encoder_output, src_mask)
        x = self.norm2(x + self.drop2(cross_attn_out))
        
        # 3. FFN
        x = self.norm3(x + self.drop3(self.ffn(x)))
        
        return x, self_attn_w, cross_attn_w


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class Transformer(nn.Module):
    """
    Full Encoder-Decoder Transformer.
    
    Architecture:
        Source tokens --> Encoder (N layers) --> Encoder representation R
        Target tokens --> Decoder (N layers, with cross-attn to R) --> Linear --> Softmax --> Predicted tokens
    """
    def __init__(self, src_vocab, tgt_vocab, d_model=512, num_heads=8,
                 num_layers=6, d_ff=2048, max_len=5000, dropout=0.1, tie_weights=True):
        super().__init__()
        self.d_model = d_model
        
        # Embeddings
        self.src_embed = nn.Embedding(src_vocab, d_model)
        self.tgt_embed = nn.Embedding(tgt_vocab, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        
        # Encoder & Decoder stacks
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)
        ])
        
        self.encoder_norm = nn.LayerNorm(d_model)
        self.decoder_norm = nn.LayerNorm(d_model)
        
        # Output projection
        self.output_proj = nn.Linear(d_model, tgt_vocab)
        
        # Weight tying (optional but common)
        if tie_weights and src_vocab == tgt_vocab:
            self.output_proj.weight = self.tgt_embed.weight
        
        self._init_weights()
    
    def _init_weights(self):
        """Xavier uniform initialization (as in original paper)."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def make_causal_mask(self, seq_len, device):
        """Upper-triangular mask: prevents attending to future positions."""
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
        return ~mask  # [seq_len, seq_len], True = allowed
    
    def encode(self, src, src_mask=None):
        x = self.pos_enc(self.src_embed(src) * np.sqrt(self.d_model))
        enc_attn_weights = []
        for layer in self.encoder_layers:
            x, attn_w = layer(x, src_mask)
            enc_attn_weights.append(attn_w)
        return self.encoder_norm(x), enc_attn_weights
    
    def decode(self, tgt, encoder_output, tgt_mask=None, src_mask=None):
        x = self.pos_enc(self.tgt_embed(tgt) * np.sqrt(self.d_model))
        dec_self_attn, dec_cross_attn = [], []
        for layer in self.decoder_layers:
            x, self_w, cross_w = layer(x, encoder_output, tgt_mask, src_mask)
            dec_self_attn.append(self_w)
            dec_cross_attn.append(cross_w)
        return self.decoder_norm(x), dec_self_attn, dec_cross_attn
    
    def forward(self, src, tgt, src_mask=None):
        # Create causal mask for decoder
        tgt_mask = self.make_causal_mask(tgt.size(1), tgt.device)
        
        # Encode source
        enc_out, enc_attn = self.encode(src, src_mask)
        
        # Decode target
        dec_out, dec_self_attn, dec_cross_attn = self.decode(tgt, enc_out, tgt_mask, src_mask)
        
        # Project to vocabulary
        logits = self.output_proj(dec_out)
        
        return logits, {
            'enc_attn': enc_attn,
            'dec_self_attn': dec_self_attn,
            'dec_cross_attn': dec_cross_attn
        }


# ============================================================
# Demo: Build and test full Transformer
# ============================================================
torch.manual_seed(42)

SRC_VOCAB = 10000
TGT_VOCAB = 10000
model = Transformer(
    src_vocab=SRC_VOCAB, tgt_vocab=TGT_VOCAB,
    d_model=256, num_heads=8, num_layers=4,
    d_ff=1024, dropout=0.1
)

# Simulate a batch: source sentence (len=12) -> target sentence (len=8)
src = torch.randint(0, SRC_VOCAB, (2, 12))   # [batch=2, src_len=12]
tgt = torch.randint(0, TGT_VOCAB, (2, 8))    # [batch=2, tgt_len=8]

logits, attn_dict = model(src, tgt)

print("=" * 60)
print("FULL TRANSFORMER MODEL")
print("=" * 60)
print(f"Source shape:      {src.shape}")
print(f"Target shape:      {tgt.shape}")
print(f"Output logits:     {logits.shape}  (batch, tgt_len, vocab)")
print(f"Encoder layers:    {len(attn_dict['enc_attn'])}")
print(f"Decoder layers:    {len(attn_dict['dec_self_attn'])}")
print(f"Enc attn shape:    {attn_dict['enc_attn'][0].shape}")
print(f"Dec self-attn:     {attn_dict['dec_self_attn'][0].shape}")
print(f"Dec cross-attn:    {attn_dict['dec_cross_attn'][0].shape}  (tgt attends to src)")

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")

In [ ]:
# ============================================================
# Attention Visualization Utility
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns
import torch

def visualize_attention(attn_weights, src_tokens=None, tgt_tokens=None,
                        layer=0, head=0, title_prefix=""):
    """
    Visualize attention weights from any attention layer.
    
    Args:
        attn_weights: list of [batch, heads, tgt_len, src_len] tensors (one per layer)
        src_tokens: list of source token strings (for axis labels)
        tgt_tokens: list of target token strings (for axis labels)
        layer: which layer to visualize
        head: which head to visualize (or 'avg' for average across heads)
    """
    attn = attn_weights[layer][0]  # remove batch dim -> [heads, tgt, src]
    
    if head == 'avg':
        attn_map = attn.mean(0).detach().cpu().numpy()
        head_label = "avg"
    else:
        attn_map = attn[head].detach().cpu().numpy()
        head_label = f"head {head}"
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(attn_map, annot=True if max(attn_map.shape) <= 12 else False,
                fmt='.2f', cmap='Blues', ax=ax, vmin=0, vmax=1,
                xticklabels=src_tokens if src_tokens else 'auto',
                yticklabels=tgt_tokens if tgt_tokens else 'auto')
    ax.set_title(f'{title_prefix}Layer {layer}, {head_label}')
    ax.set_xlabel('Key (attending TO)')
    ax.set_ylabel('Query (attending FROM)')
    plt.tight_layout()
    plt.show()


def visualize_all_heads(attn_weights, layer=0, src_tokens=None, tgt_tokens=None,
                        title_prefix=""):
    """Show all attention heads for a given layer in a grid."""
    attn = attn_weights[layer][0].detach().cpu().numpy()  # [heads, tgt, src]
    num_heads = attn.shape[0]
    cols = min(4, num_heads)
    rows = (num_heads + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
    if num_heads == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for h in range(num_heads):
        sns.heatmap(attn[h], cmap='Blues', ax=axes[h], vmin=0, vmax=1,
                    xticklabels=src_tokens if src_tokens else False,
                    yticklabels=tgt_tokens if tgt_tokens else False,
                    cbar=False)
        axes[h].set_title(f'Head {h}', fontsize=10)
    
    for h in range(num_heads, len(axes)):
        axes[h].set_visible(False)
    
    fig.suptitle(f'{title_prefix}Layer {layer} - All Heads', fontsize=13)
    plt.tight_layout()
    plt.show()


# === Demo with our Transformer model ===
src_words = ["I", "am", "a", "good", "student", "<pad>"] * 2
tgt_words = ["<sos>", "Je", "suis", "bon", "<pad>"] + ["<sos>", "a", "b", "<pad>"]

# Use the model from previous cell
print("Encoder Self-Attention (Layer 0, all heads):")
visualize_all_heads(attn_dict['enc_attn'], layer=0, title_prefix="Encoder ")

print("\nDecoder Cross-Attention (Layer 0) -- how decoder attends to encoder:")
visualize_attention(attn_dict['dec_cross_attn'], layer=0, head='avg',
                    title_prefix="Decoder Cross-Attn: ")

print("\nDecoder Masked Self-Attention (Layer 0) -- causal pattern visible:")
visualize_attention(attn_dict['dec_self_attn'], layer=0, head=0,
                    title_prefix="Decoder Masked Self-Attn: ")

# Transformer Cheat Sheet: FFN, Multi-Head, Masked Multi-Head

> **Core idea**
>
> - **Attention** = tokens **communicate with other tokens**
> - **FFN** = each token **processes what it learned privately**
> - **Multi-head** = attention happens in **multiple parallel views**
> - **Masked multi-head** = same as multi-head, but **future tokens are blocked**

---

## 1) Transformer block at a glance

### Encoder block
1. Multi-Head **Self-Attention**
2. Add + Norm
3. **Feed Forward Network (FFN)**
4. Add + Norm

### Decoder block
1. **Masked** Multi-Head Self-Attention
2. Add + Norm
3. Cross-Attention (to encoder output)
4. Add + Norm
5. **Feed Forward Network (FFN)**
6. Add + Norm

---
> Add + Norm means residual connection followed by layer normalization. The residual connection adds the input back to the sublayer output so the model preserves original information and improves gradient flow. Layer normalization then stabilizes the activations, making training deeper Transformer networks easier and more stable. 
## 2) What exactly does FFN do?

FFN = **Feed Forward Network**  
Also called **Position-wise Feed Forward Network**

It is applied to **each token independently** after attention.

### Formula
\[
FFN(x) = W_2 \, \sigma(W_1x + b_1) + b_2
\]

Usually:

\[
d_{model} \rightarrow d_{ff} \rightarrow d_{model}
\]

Example:

\[
512 \rightarrow 2048 \rightarrow 512
\]

### What it means
After attention gives a token **context**, FFN **refines** that token’s representation.

### Important
- FFN does **not** mix information across tokens
- FFN uses the **same weights for every token position**
- Token interaction happens in **attention**, not FFN

### Best interview line
> Self-attention mixes information across tokens, while the feed-forward network applies nonlinear transformation independently to each token.

---

## 3) What does attention do?

Each token creates:

- **Q (Query)** = what am I looking for?
- **K (Key)** = what do I contain?
- **V (Value)** = what information can I pass?

### Core formula
\[
Attention(Q, K, V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V
\]

### Meaning
A token asks:
> Which other tokens matter to me, and how much?

So attention is where **token-to-token interaction** happens.

---

## 4) What is Multi-Head Attention?

Instead of doing attention once, we do it **multiple times in parallel**.

Each head can learn a different relation, such as:
- local context
- subject-object relation
- long-range dependency
- semantic similarity

Then all head outputs are:
1. concatenated
2. projected with a final linear layer

### Why multiple heads?
Because one head may capture only one kind of pattern.  
Multiple heads let the model learn **different relationships at the same time**.

### Best interview line
> Multi-head attention lets the model attend to the sequence from multiple learned representation subspaces in parallel.

---

## 5) What is Masked Multi-Head Attention?

It is still **multi-head attention**, but with a **mask** applied.

### Purpose
Prevent a token from seeing **future tokens**

In decoder generation, token at position `t` can only attend to positions `<= t`.

### Example
For sequence:

`I love deep learning`

When predicting `deep`, the model can look at:
- `I`
- `love`

But it **cannot** look at:
- `learning`

That would leak future information.

### Best interview line
> Masked multi-head attention is standard multi-head self-attention with a causal mask that blocks access to future positions.

---

## 6) Multi-Head vs Masked Multi-Head

| Component | What it does | Future tokens visible? |
|---|---|---|
| Multi-Head Self-Attention | attends in multiple parallel ways | Usually yes in encoder |
| Masked Multi-Head Self-Attention | same, but with causal mask | No |

### Key distinction
The difference is **not** number of heads.  
The difference is whether a **mask** blocks illegal positions.

---

## 7) Encoder vs Decoder

| Part | Attention type | Why |
|---|---|---|
| Encoder | Multi-Head Self-Attention | Can see full input |
| Decoder | Masked Multi-Head Self-Attention | Cannot peek at future output |
| Decoder | Cross-Attention | Looks at encoder output |

---

## 8) Tiny flow example

Sentence:

`I love pizza`

### Step 1: Embedding
- `I` -> x1
- `love` -> x2
- `pizza` -> x3

### Step 2: Self-attention
Each token looks at all tokens and becomes context-aware:
- `I` learns it is subject
- `love` learns relation to subject and object
- `pizza` learns it is object

### Step 3: FFN
Each token is refined independently:
- FFN(h1)
- FFN(h2)
- FFN(h3)

### Mental model
- **Attention** = collect context
- **FFN** = process context

---

## 9) One-line memory tricks

- **Attention** = look around
- **Multi-head** = look around in many ways
- **Masked attention** = look only backward
- **FFN** = think privately

---

## 10) Most important interview distinctions

### Where does token mixing happen?
In **attention**

### Does FFN mix tokens?
No

### Why do we need FFN if we already have attention?
Attention gathers context; FFN adds **nonlinear feature transformation per token**

### Why is masking needed?
To prevent **future-token leakage** during autoregressive generation

---

## 11) 30-second interview answer

> A Transformer block has attention and feed-forward sublayers. Self-attention lets each token gather information from other tokens. Multi-head attention does this through several heads in parallel, allowing the model to learn different relationships at the same time. In the decoder, masked multi-head attention adds a causal mask so a token cannot attend to future tokens. After attention, each token passes through the same feed-forward network independently, which refines the token representation without mixing information across tokens.

---

## 12) Mermaid diagram: full transformer view

```mermaid
flowchart TD
    A[Input Tokens] --> B[Embeddings + Positional Encoding]

    B --> C[Multi-Head Self-Attention]
    C --> D[Add + Norm]
    D --> E[Feed Forward Network]
    E --> F[Add + Norm]
    F --> G[Encoder Output]

    B --> H[Masked Multi-Head Self-Attention]
    H --> I[Add + Norm]
    G --> J[Cross-Attention]
    I --> J
    J --> K[Add + Norm]
    K --> L[Feed Forward Network]
    L --> M[Add + Norm]
    M --> N[Decoder Output]

---

# 6. Training & Optimization

## 6.1 Loss Function: Cross-Entropy with Label Smoothing

**Standard cross-entropy** for next-token prediction:

$$\mathcal{L} = -\sum_{t=1}^{T} \log P(y_t | y_{<t}, X)$$

**Label smoothing** (used in original paper, $\epsilon = 0.1$): Instead of the one-hot target $[0, 0, 1, 0, ...]$, use $[0.1/V, 0.1/V, 0.9 + 0.1/V, 0.1/V, ...]$

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Why label smoothing helps:</b>
<ul>
<li>Prevents the model from becoming over-confident (outputting extreme logits)</li>
<li>Acts as a regularizer -- improves generalization</li>
<li>Slightly hurts perplexity but improves BLEU score (better actual translations)</li>
</ul>
</div>

---

## 6.2 Learning Rate Schedule: Warmup + Decay

The original Transformer uses a distinctive learning rate schedule:

$$lr = d_{\text{model}}^{-0.5} \cdot \min(\text{step}^{-0.5}, \; \text{step} \cdot \text{warmup\_steps}^{-1.5})$$

This creates two phases:
1. **Warmup** (first ~4000 steps): LR increases linearly from 0
2. **Decay** (after warmup): LR decreases proportional to $1/\sqrt{\text{step}}$

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Why warmup is needed:</b>
<ul>
<li>Early in training, gradients are unreliable (random weights, high variance in Adam's second moment estimates)</li>
<li>A large LR at the start causes divergence -- warmup lets Adam accumulate stable statistics first</li>
<li>Post-LN architectures are especially sensitive; Pre-LN models can often skip warmup</li>
</ul>
</div>

**Modern alternative:** Cosine annealing with warmup (used in GPT-3, LLaMA):
$$lr = lr_{\max} \cdot 0.5 \cdot (1 + \cos(\pi \cdot \text{progress}))$$

---

## 6.3 Gradient Clipping

```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

<div style="background-color: #fcf8e3; border: 1px solid #faebcc; color: #8a6d3b; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Why:</b> Transformers are prone to gradient explosion, especially with Post-LN and long sequences. Clipping ensures the total gradient norm never exceeds <code>max_norm</code>, preventing catastrophic parameter updates. Typical value: 1.0.
</div>

---

## 6.4 Mixed Precision Training (FP16 / BF16)

| Precision | Bits | Range | Used For |
|---|---|---|---|
| FP32 | 32 | Very wide | Master weights, loss scaling |
| FP16 | 16 | Limited (needs loss scaling) | Forward/backward pass, NVIDIA GPUs |
| BF16 | 16 | Same range as FP32 | Forward/backward pass, newer GPUs (A100+) |

**How it works:**
1. Keep a **master copy** of weights in FP32
2. Cast to FP16/BF16 for forward and backward pass (2x faster, 50% less memory)
3. Compute gradients in FP16/BF16
4. Update master weights in FP32 (for numerical stability)

```python
# PyTorch automatic mixed precision
scaler = torch.cuda.amp.GradScaler()
with torch.cuda.amp.autocast():
    logits = model(src, tgt)
    loss = criterion(logits, targets)
scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()
```

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>BF16 vs FP16:</b> BF16 has the same exponent range as FP32 (8 bits), so it almost never overflows/underflows. FP16 has only 5 exponent bits and requires careful loss scaling. <b>Prefer BF16</b> when hardware supports it (A100, H100, TPUs).
</div>

---

## 6.5 Gradient Checkpointing (Activation Recomputation)

**Problem:** Storing activations for backprop in a 175B parameter model requires hundreds of GB of GPU memory.

**Solution:** Do not store intermediate activations. Instead, re-compute them during the backward pass.

- **Trade-off:** ~33% more computation, but ~60-70% less activation memory
- **Implementation:** `torch.utils.checkpoint.checkpoint(layer, input)`
- **Used by:** Nearly all large model training (GPT-3, LLaMA, etc.)

---

## 6.6 Parallelism Strategies

| Strategy | What is split | Communication | Memory saving |
|---|---|---|---|
| **Data Parallelism** (DDP) | Batch across GPUs | Gradient all-reduce | None (full model per GPU) |
| **Tensor Parallelism** (TP) | Weight matrices within a layer | Heavy (per-layer all-reduce) | Linear with #GPUs |
| **Pipeline Parallelism** (PP) | Layers across GPUs | Only at stage boundaries | Linear with #stages |
| **FSDP / ZeRO** | Optimizer states + gradients + weights | All-gather before forward | Up to linear with #GPUs |

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>In practice:</b> Large model training combines multiple strategies. Example (LLaMA 70B): TP=8 within a node, PP=4 across nodes, FSDP for optimizer states. This is called <b>3D parallelism</b>.
</div>

---

## 6.7 Weight Initialization

| Component | Strategy | Rationale |
|---|---|---|
| Embeddings | $\mathcal{N}(0, 1)$ or Xavier | Standard for lookup tables |
| Attention W_q, W_k, W_v, W_o | Xavier Uniform | Maintains variance through projections |
| FFN layers | Xavier Uniform or Kaiming (with ReLU) | Matches activation function |
| LayerNorm $\gamma$ | Ones | Start as identity |
| LayerNorm $\beta$ | Zeros | No initial shift |
| Output residual layers | Scaled by $1/\sqrt{2N}$ | GPT-2 trick: scale down residual contributions in deeper layers |

---

# 7. Modern Architectures & Interview Prep

## 7.1 The Three Transformer Variants

| | Encoder-Only | Decoder-Only | Encoder-Decoder |
|---|---|---|---|
| **Example** | BERT, RoBERTa, DeBERTa | GPT-2/3/4, LLaMA, Mistral, Claude | T5, BART, mBART |
| **Attention** | Bidirectional (full) | Causal (masked) | Encoder: full; Decoder: causal + cross |
| **Pre-training** | Masked Language Model (MLM) | Autoregressive LM (next token) | Span corruption / denoising |
| **Strengths** | Understanding, classification, retrieval | Generation, zero/few-shot, reasoning | Seq2seq (translation, summarization) |
| **Input/Output** | Single sequence in, embedding out | Prefix in, continuation out | Source in, target out |

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Industry trend (2023-2025):</b> Decoder-only models have become dominant for general-purpose AI. Encoder-only models remain strong for embedding/retrieval tasks. Encoder-decoder models are niche but excel at structured generation (translation, summarization with specific format).
</div>

---

## 7.2 KV-Cache for Efficient Inference

**Problem:** In autoregressive generation, at step $t$ we compute attention over all positions $1 \ldots t$. Without caching, this means re-computing K and V for all previous tokens at every step -- $O(T^2)$ total computation.

**Solution: KV-Cache**

At each generation step:
1. Only compute Q, K, V for the **new token** (the one just generated)
2. **Append** the new K and V to the cached K and V from all previous steps
3. Compute attention: new Q attends to the full cached K, V

```
Step 1: Q1, K1, V1 --> cache = {K: [K1], V: [V1]}
Step 2: Q2, K2, V2 --> cache = {K: [K1, K2], V: [V1, V2]}
        Attention: Q2 @ [K1, K2]^T --> softmax --> @ [V1, V2]
Step 3: Q3, K3, V3 --> cache = {K: [K1, K2, K3], V: [V1, V2, V3]}
        ...
```

<div style="background-color: #fcf8e3; border: 1px solid #faebcc; color: #8a6d3b; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Memory cost:</b> KV-cache grows linearly with sequence length: <code>2 * num_layers * num_heads * d_head * seq_len * bytes_per_element</code>. For a 70B model generating 4K tokens in BF16, this is ~40GB. This is often the bottleneck for long-context inference.
<br><br>
<b>Optimizations:</b> Multi-Query Attention (MQA) and Grouped-Query Attention (GQA) reduce KV-cache by sharing K/V across heads. GQA (used in LLaMA 2 70B) uses 8 KV heads instead of 64, reducing cache by 8x.
</div>

---

## 7.3 Speculative Decoding

**Problem:** Autoregressive decoding is slow because each token requires a full forward pass through the large model, and these passes are sequential (cannot be parallelized).

**Solution:** Use a small, fast **draft model** to generate $k$ candidate tokens in parallel, then **verify** them all at once with the large model in a single forward pass.

**How it works:**
1. **Draft phase:** Small model (e.g., 1B params) generates $k$ tokens autoregressively (fast, ~5x faster per token)
2. **Verify phase:** Large model (e.g., 70B params) processes all $k$ tokens in **one** forward pass and checks which ones it agrees with
3. **Accept/reject:** Accept the longest prefix of tokens where both models agree (using rejection sampling to maintain exact distribution of the large model)
4. **Repeat** from the first rejected position

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Key insight:</b> Speculative decoding produces the <b>exact same distribution</b> as the large model alone -- it is a lossless speedup. Typical speedup: 2-3x for well-matched draft/target model pairs.
</div>

---

## 7.4 Other Modern Optimizations (Brief)

| Technique | What | Impact |
|---|---|---|
| **Flash Attention** | Fused CUDA kernel, tiling, never materializes full attention matrix | 2-4x faster attention, enables longer contexts |
| **Grouped-Query Attention (GQA)** | Share K/V across groups of query heads | Reduces KV-cache, nearly matches MHA quality |
| **Sliding Window Attention** | Each token attends only to last $w$ tokens (Mistral) | O(n*w) instead of O(n^2), handles long contexts |
| **Mixture of Experts (MoE)** | Only activate subset of FFN parameters per token | More params, same compute (Mixtral, GPT-4) |
| **RMSNorm** | Simplified LayerNorm (no mean centering) | 10-15% faster normalization |
| **SwiGLU** | Gated FFN activation | Better quality than ReLU/GELU |

---

## 7.5 Interview Questions with Answer Hints

### Basic Level

**Q1: Why do transformers need positional encoding?**
> Transformers process all tokens in parallel (no recurrence), so they have no inherent notion of token order. PE injects position information. Without it, "dog bites man" = "man bites dog".

**Q2: What is the purpose of the scaling factor $\sqrt{d_k}$ in attention?**
> As $d_k$ grows, dot products grow in magnitude, pushing softmax into regions with tiny gradients (saturation). Dividing by $\sqrt{d_k}$ keeps the variance of scores ~1, maintaining useful gradient flow.

**Q3: Why multi-head attention instead of single-head?**
> Different heads can attend to different types of relationships (syntactic, semantic, positional). A single head must compress all relationship types into one attention distribution.

**Q4: What are residual connections and why are they needed?**
> Skip connections that add the input to the output of each sublayer: `output = x + sublayer(x)`. They enable gradient flow through deep networks and make it easier to learn (sublayer only needs to learn the residual/delta).

### Intermediate Level

**Q5: Explain the difference between Pre-LN and Post-LN transformers.**
> Post-LN: `LN(x + sublayer(x))` -- original, requires warmup, can achieve slightly better final quality. Pre-LN: `x + sublayer(LN(x))` -- modern default, much more stable training, no warmup needed, easier to scale to large models.

**Q6: How does cross-attention work in the decoder?**
> Q comes from the decoder (current target representation), K and V come from the encoder output. This lets each target position "look at" all source positions to decide what to translate/generate next.

**Q7: What is teacher forcing and what problem does it cause?**
> During training, the decoder sees the ground-truth previous tokens. At inference, it sees its own (potentially wrong) predictions. This train/test mismatch is called "exposure bias." Mitigations: scheduled sampling, reinforcement learning fine-tuning.

**Q8: Compare RoPE and ALiBi for positional encoding.**
> RoPE rotates Q/K vectors by position-dependent angles; relative position emerges from the dot product. ALiBi adds a linear distance penalty directly to attention scores. ALiBi extrapolates better out-of-the-box; RoPE is more widely adopted and extrapolates well with scaling techniques (NTK, YaRN).

### Advanced Level

**Q9: How does KV-cache work and what are its memory implications?**
> During autoregressive generation, K and V for previously generated tokens are cached rather than recomputed. Memory grows as `O(layers * heads * d_head * seq_len)`. For long contexts, KV-cache becomes the memory bottleneck. GQA/MQA reduce this by sharing K/V across query heads.

**Q10: Explain speculative decoding. Does it change the output distribution?**
> A small draft model generates $k$ candidate tokens quickly. The large target model verifies all $k$ in one pass. Accepted tokens are exact samples from the target model distribution (proven via modified rejection sampling). Net effect: lossless 2-3x speedup.

**Q11: Why did the field converge on decoder-only architectures for LLMs?**
> (1) Simpler architecture (no encoder, no cross-attention). (2) Autoregressive training is a single unified objective. (3) Scales well -- same architecture works for all tasks via prompting. (4) In-context learning emergent at scale. (5) Encoder-decoder has an awkward boundary between "understanding" and "generation" for open-ended tasks.

**Q12: What is Flash Attention and why is it important?**
> Standard attention materializes the full $N \times N$ attention matrix in GPU HBM (slow, memory-hungry). Flash Attention uses tiling to compute attention block-by-block in fast SRAM, never materializing the full matrix. Result: exact same output, 2-4x faster, memory usage goes from $O(N^2)$ to $O(N)$.

**Q13: Explain the difference between Tensor Parallelism, Pipeline Parallelism, and FSDP.**
> TP splits individual weight matrices across GPUs (column/row splitting), requires all-reduce per layer. PP puts different layers on different GPUs, requires micro-batching to avoid bubble overhead. FSDP (ZeRO) shards optimizer states/gradients/weights across GPUs but all-gathers full weights before each forward pass. Large-scale training combines all three.

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Interview tip:</b> For architecture questions, always be ready to (1) draw the data flow, (2) state the shapes at each step, (3) explain <i>why</i> each component exists (what breaks if you remove it), and (4) name which real models use which variant.
</div>

---

# 2. All Attention Variants (End-to-End)

## 2.1 Multi-Head Attention (MHA)

<div style="background-color: #fcf8e3; border: 1px solid #faebcc; color: #8a6d3b; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Problem:</b> A single attention head compresses all types of relationships (syntactic, semantic, positional) into one attention distribution. It may be dominated by one pattern.
</div>

**Solution:** Run **h parallel attention heads**, each with its own learned W_Q, W_K, W_V projections operating on a smaller dimension ($d_k = d_{model}/h$). Then **concatenate** all outputs and apply a final projection:

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, \ldots, \text{head}_h) \cdot W_O$$
$$\text{where head}_i = \text{Attention}(QW_Q^i, KW_K^i, VW_V^i)$$

![Multi-head attention formula](https://learning.oreilly.com/api/v2/epubs/urn:orm:book:9781838821593/files/assets/15ea0152-cd90-4ffd-832a-09e5b931e185.png)

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Why multiple heads work:</b> Different heads learn to attend to different things — one head might track syntactic subject-verb relationships, another might track coreference, another might attend to nearby words for local context. The concatenation + projection combines these diverse perspectives into a richer representation.
</div>

| Aspect | Detail |
|---|---|
| **Complexity** | O(n²d) — same total FLOPs as single-head (split across heads) |
| **Parameters** | $W_Q, W_K, W_V \in \mathbb{R}^{d \times d}$ each + $W_O \in \mathbb{R}^{d \times d}$ |
| **KV-Cache per token** | $2 \times h \times d_k$ elements (separate K, V per head) |
| **Used by** | Original Transformer, BERT, GPT-2, T5 |

---

## 2.2 Masked / Causal Self-Attention

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Problem:</b> In autoregressive (left-to-right) generation, token at position <i>t</i> must NOT see tokens at positions <i>t+1, t+2, ...</i> That would be looking at the future — cheating!
</div>

**Solution:** Before softmax, set all "future" positions to $-\infty$:

$$\text{score}_{ij} = \begin{cases} \frac{q_i \cdot k_j}{\sqrt{d_k}} & \text{if } j \leq i \\ -\infty & \text{if } j > i \end{cases}$$

Since $\text{softmax}(-\infty) = 0$, future tokens receive **zero attention weight** — they are invisible.

The mask is a **lower-triangular matrix** of ones — position $i$ can only attend to positions $\leq i$.

| Aspect | Detail |
|---|---|
| **Core idea** | Prevent attending to future positions via $-\infty$ masking |
| **Complexity** | O(n²d) — same as unmasked, but half the scores are zeroed |
| **Used by** | GPT family, LLaMA, Mistral, Claude — all autoregressive models |

---

## 2.3 Cross-Attention (Encoder-Decoder Attention)

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Problem:</b> The decoder needs to "look back" at the source sequence (encoder output). Self-attention only attends within the same sequence.
</div>

**Solution:** Q comes from the **decoder**, K and V come from the **encoder**:

$$Q = M \cdot W_Q \quad \text{(decoder hidden state)} \qquad K = R \cdot W_K, \; V = R \cdot W_V \quad \text{(encoder output)}$$

The attention matrix is **rectangular** `[tgt_len × src_len]` — each target token attends to all source tokens.

| Aspect | Detail |
|---|---|
| **Core idea** | Bridge between encoder and decoder; Q from one sequence, K/V from another |
| **Complexity** | O($n_t \cdot n_s \cdot d$) where $n_t$ = target length, $n_s$ = source length |
| **Used by** | T5, BART, Whisper, Stable Diffusion (cross-attends to text embeddings) |

---

## 2.4 Multi-Query Attention (MQA)

**Paper:** "Fast Transformer Decoding" (Shazeer, 2019) &nbsp; | &nbsp; **Used in:** PaLM, Falcon

<div style="background-color: #fcf8e3; border: 1px solid #faebcc; color: #8a6d3b; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Problem:</b> KV-cache in standard MHA stores separate K, V for <b>every</b> head. For a model with 64 heads, that is 64 K vectors + 64 V vectors per token per layer. At long context, this dominates GPU memory.
</div>

**Solution:** Keep **separate Q per head**, but use a **single shared K and single shared V** across all heads:

- Q: $h$ separate projections (as in MHA)
- K: **1** shared projection (broadcast to all heads)
- V: **1** shared projection (broadcast to all heads)

| Aspect | Detail |
|---|---|
| **KV-Cache reduction** | $h\times$ smaller than MHA (e.g., 64x for 64 heads) |
| **Quality** | Slight degradation vs MHA, but often acceptable |
| **Why it works** | Queries need to be diverse (different heads ask different questions), but keys/values can be shared without much quality loss |
| **Used by** | PaLM, Falcon, StarCoder |

---

## 2.5 Grouped-Query Attention (GQA)

**Paper:** "GQA: Training Generalized Multi-Query Transformer Models" (Ainslie et al., 2023)

**Used in:** LLaMA 2/3, Mistral, Mixtral, Gemma 2, Qwen 2

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>The sweet spot between MHA and MQA.</b> Instead of 1 shared KV (MQA) or h separate KVs (MHA), use <b>g groups</b>. Each group of $h/g$ query heads shares one K and one V.
</div>

**Example (LLaMA 2 70B):** 64 query heads, 8 KV groups → each group of 8 query heads shares 1 K and 1 V.

| Setting | KV heads | Cache size | Quality |
|---|---|---|---|
| MHA | h = 64 | 64 K + 64 V | Best quality |
| **GQA (g=8)** | **8** | **8 K + 8 V** | **Near-MHA quality** |
| MQA (g=1) | 1 | 1 K + 1 V | Some quality loss |

| Aspect | Detail |
|---|---|
| **Core idea** | g groups of query heads share K/V — interpolate between MHA and MQA |
| **KV-Cache** | $h/g$ times smaller than MHA |
| **Why it won** | MQA is too aggressive for large models; GQA recovers most MHA quality with most MQA efficiency |
| **Used by** | LLaMA 2 70B (g=8), LLaMA 3, Mistral 7B, Mixtral, Gemma 2 |

---

## 2.6 Multi-Head Latent Attention (MLA)

**Paper:** "DeepSeek-V2" (2024) &nbsp; | &nbsp; **Used in:** DeepSeek-V2, DeepSeek-V3, DeepSeek-R1

<div style="background-color: #d9edf7; border: 1px solid #bce8f1; color: #31708f; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Problem:</b> Even GQA's KV-cache is large for very long contexts (100K+ tokens). Can we compress it further without losing quality?
</div>

**Solution:** Compress K and V into a **single low-rank latent vector** $c_t$ per token:

$$c_t = W_{\text{down}} \cdot x_t \quad \text{(compress: } d_{model} \to d_c \text{, where } d_c \ll d_{model}\text{)}$$
$$K_t = W_{\text{up}}^K \cdot c_t, \quad V_t = W_{\text{up}}^V \cdot c_t \quad \text{(expand back at attention time)}$$

**Key insight:** Only cache $c_t$ (small), not full K and V. Reconstruct K, V on-the-fly during attention.

| Aspect | Detail |
|---|---|
| **KV-Cache** | Only $d_c$ elements per token (vs $2 \times h \times d_k$ for MHA). Example: 512 vs 8192 elements |
| **Quality** | Matches or exceeds MHA — the low-rank bottleneck acts as regularization |
| **Trade-off** | Slightly more computation (decompression), but dramatically less memory |
| **Used by** | DeepSeek-V2/V3/R1 |

---

## 2.7 Linear Attention

**Paper:** "Transformers are RNNs" (Katharopoulos et al., 2020)

<div style="background-color: #fcf8e3; border: 1px solid #faebcc; color: #8a6d3b; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Problem:</b> Standard attention is O(n²) in sequence length. For sequences of 10K-100K+ tokens, this becomes prohibitive.
</div>

**Solution:** Replace softmax with a **kernel function** $\phi$, then exploit associativity of matrix multiplication to change the order of operations:

**Standard:** $\text{softmax}(QK^T) \cdot V$ → compute $n \times n$ matrix first → **O(n²d)**

**Linear:** $\phi(Q) \cdot (\phi(K)^T \cdot V)$ → compute $d \times d$ matrix first → **O(nd²)**

When $d \ll n$ (long sequences), this is dramatically faster.

| Aspect | Detail |
|---|---|
| **Complexity** | O(nd²) instead of O(n²d) |
| **Quality trade-off** | Approximation — loses the sharp attention patterns softmax provides; often lower quality on tasks needing precise retrieval |
| **Used by** | Linear Transformer, (RWKV related concepts) |

---

## 2.8 Sparse Attention

**Papers:** Sparse Transformer (2019), Longformer (2020), BigBird (2020)

**Solution:** Instead of attending to ALL positions, attend only to a carefully chosen **subset** using sparse patterns:

- **Local/Sliding window:** Each token attends to $w$ neighbors on each side — captures local context
- **Strided/Dilated:** Attend to every k-th token — captures periodic/structural patterns
- **Global tokens:** A few special tokens (e.g., `[CLS]`) attend to everything and everything attends to them
- **Random:** Randomly selected positions — provably helps with graph connectivity

**Longformer** = local + global. **BigBird** = local + global + random (proven universal approximator).

| Aspect | Detail |
|---|---|
| **Complexity** | O(n·w) for window size w, or O(n·√n) for strided |
| **Quality** | Works well for tasks where local context dominates |
| **Used by** | Longformer, BigBird, LED |

---

## 2.9 Flash Attention

**Paper:** "FlashAttention" (Dao et al., 2022, 2023)

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Critical distinction:</b> Flash Attention is <b>NOT an approximation</b>. It computes the <b>exact same result</b> as standard attention but uses an IO-aware algorithm that is 2-4x faster and uses O(n) memory instead of O(n²).
</div>

**Problem:** Standard attention is **memory-bound**, not compute-bound. The bottleneck is reading/writing the $n \times n$ attention matrix to GPU HBM (high-bandwidth memory).

**Solution — Tiling + Online Softmax:**

1. **Tile** Q, K, V into blocks that fit in **SRAM** (fast on-chip memory, ~20MB)
2. For each block of Q, iterate over blocks of K, V:
   - Compute partial attention scores in SRAM
   - Use **online softmax** trick to accumulate the correct normalization incrementally
   - Write only the final output block to HBM
3. The full $n \times n$ matrix **never exists in HBM**

| Aspect | Detail |
|---|---|
| **Memory** | O(n) instead of O(n²) — the n×n matrix never fully materializes |
| **Speed** | 2-4x faster than standard PyTorch attention |
| **Exactness** | **Bit-for-bit identical** to standard attention |
| **Flash Attention 2** | Optimized for A100/H100 with better parallelism |
| **Flash Attention 3** | Targets H100, uses FP8 tensor cores, async execution |
| **Used by** | Virtually every modern LLM: LLaMA 2/3, Mistral, GPT-4, Claude |

<div style="background-color: #fcf8e3; border: 1px solid #faebcc; color: #8a6d3b; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Interview must-know:</b> "Flash Attention is not a new attention mechanism — it is an <i>implementation optimization</i> of standard scaled dot-product attention. It produces the exact same output. The innovation is in how memory access patterns exploit GPU memory hierarchy."
</div>

---

## 2.10 Sliding Window Attention (SWA)

**Used in:** Mistral 7B, Mixtral, Gemma 2

Each token attends only to the previous $w$ tokens (a fixed window). **But doesn't this lose long-range info?** No — through **stacking**. If each layer has window $w$, then after $L$ layers, information propagates up to $L \times w$ tokens. With $w=4096$ and $L=32$, effective receptive field = 131K tokens.

| Aspect | Detail |
|---|---|
| **Complexity** | O(n·w·d) per layer instead of O(n²·d) |
| **KV-cache** | Only cache last $w$ tokens (rolling buffer) — constant memory! |
| **Used by** | Mistral 7B (w=4096), Mixtral, Gemma 2 (alternates SWA with full attention) |

---

## 2.11 Ring Attention

**Paper:** "Ring Attention with Blockwise Transformers" (Liu et al., 2023)

**Problem:** For extremely long sequences (1M+ tokens), even Flash Attention can't fit on a single GPU.

**Solution:** Distribute the sequence across devices in a **ring topology**:

1. Each device holds a **shard** of Q tokens
2. K, V blocks are **rotated around the ring** — at each step, each device computes attention with the current K, V block, then passes it to the next device
3. Communication (sending K, V) **overlaps** with computation
4. After one full rotation, every Q has attended to every K, V

```
Device 0: Q[0:1024]     ←─ K,V blocks rotate through ring
Device 1: Q[1024:2048]
Device 2: Q[2048:3072]
Device 3: Q[3072:4096]
```

| Aspect | Detail |
|---|---|
| **Complexity** | Same total FLOPs, distributed across devices |
| **Used by** | Research systems for million-token contexts, long-context LLaMA training |

---

## 2.12 Differential Attention

**Paper:** "Differential Transformer" (Ye et al., Microsoft Research, 2024)

**Problem:** Standard attention maps are noisy — many tokens get small but non-zero weights, creating a "noise floor" that dilutes focus.

**Solution:** Use **two separate softmax attention maps** and take their **difference** to cancel noise:

$$\text{DiffAttn}(Q, K, V) = \left(\text{softmax}\!\left(\frac{Q_1 K_1^T}{\sqrt{d}}\right) - \lambda \cdot \text{softmax}\!\left(\frac{Q_2 K_2^T}{\sqrt{d}}\right)\right) V$$

- Split Q into $Q_1, Q_2$ and K into $K_1, K_2$ (halve head dimension)
- $\lambda$ is a learnable scalar per head
- Common noise in both maps **cancels out**; signal remains

**Analogy:** Like noise-canceling headphones — two microphones pick up ambient noise, subtracting them cancels noise and preserves signal.

| Aspect | Detail |
|---|---|
| **Quality** | Outperforms standard Transformer at same model size |
| **Used by** | Differential Transformer (Microsoft, 2024); early adoption |

---

## 2.13 Comparison Table: All Attention Variants

| Variant | Time | Space | KV-Cache | Exact? | Key Models |
|---|---|---|---|---|---|
| **Scaled Dot-Product** | O(n²d) | O(n²) | 2hnd_k | Yes | Original Transformer |
| **Multi-Head (MHA)** | O(n²d) | O(n²h) | 2hnd_k | Yes | BERT, GPT-2, T5 |
| **Masked/Causal** | O(n²d) | O(n²) | 2hnd_k | Yes | GPT, LLaMA, Claude |
| **Cross-Attention** | O(n_t·n_s·d) | O(n_t·n_s) | 2hn_s·d_k | Yes | T5, BART, Whisper |
| **MQA** | O(n²d) | O(n²) | **2nd_k** | Yes | PaLM, Falcon |
| **GQA** | O(n²d) | O(n²) | **2gnd_k** | Yes | LLaMA 2/3, Mistral |
| **MLA** | O(n²d) | O(n²) | **nd_c** | Yes | DeepSeek-V2/V3/R1 |
| **Linear** | **O(nd²)** | **O(nd)** | O(d²) | No | Linear Transformer |
| **Sparse** | **O(nwd)** | O(nw) | Windowed | Yes* | Longformer, BigBird |
| **Flash Attention** | O(n²d) | **O(n)** | 2hnd_k | **Yes** | LLaMA, Mistral, GPT-4 |
| **Sliding Window** | **O(nwd)** | O(nw) | **2hwd_k** | Yes* | Mistral, Gemma 2 |
| **Ring Attention** | O(n²d) | O(n²/p) | Distributed | Yes | Long-context training |
| **Differential** | O(n²d) | O(n²) | 2hnd_k | Yes | Diff Transformer (2024) |

*Sparse/SWA compute exact attention within their pattern but skip positions outside it.

<div style="background-color: #dff0d8; border: 1px solid #d0e9c6; color: #3c763d; padding: 10px; border-radius: 5px; margin: 10px 0;">
<b>Modern LLM recipe (2024-2025):</b> <b>GQA + RoPE + Causal Masking + Flash Attention</b> + (optionally) Sliding Window in some layers. This is what LLaMA 3, Mistral, Gemma 2, and most production LLMs use.
</div>

---

# 8. Rapid-Fire Transformer Interview Q&A

<div style="background-color: #f0f7ff; border: 2px solid #3498db; padding: 20px; border-radius: 10px; margin: 10px 0;">

### Q1: What is self-attention and why is it needed?
**A:** Self-attention allows each token in a sequence to attend to every other token, computing a weighted sum of all value vectors based on relevance scores. It is needed because, unlike RNNs, it captures **long-range dependencies in O(1) sequential operations** and enables **parallel computation** over the entire sequence. Each token computes: Attention(Q,K,V) = softmax(QKᵀ/√d_k)V, where the attention weights dynamically determine how much each token "looks at" every other token.

---

### Q2: What is the complexity of self-attention?
**A:** The time and memory complexity is **O(n²d)** where n is sequence length and d is the dimension. The n² comes from computing the attention score between every pair of tokens (the QKᵀ matrix is n×n). This quadratic scaling is the main bottleneck for long sequences, motivating variants like **Linformer (O(nd))**, **Performer (O(nd))**, **Sparse Attention (O(n√n))**, and **Flash Attention** (which is exact but IO-aware, reducing memory from O(n²) to O(n)).

---

### Q3: Why multiply Q·Kᵀ then divide by √d_k?
**A:** The dot product Q·Kᵀ measures similarity between queries and keys, but as the dimension d_k grows, the dot product magnitudes grow proportionally (variance ≈ d_k for random vectors). Large magnitudes push softmax into regions with **extremely small gradients** (saturation), making training unstable. Dividing by √d_k normalizes the variance back to ~1, keeping the softmax in a well-behaved gradient region. This is why it is called **"scaled" dot-product attention**.

---

### Q4: What is multi-head attention and why use it?
**A:** Instead of one attention function with d_model dimensions, multi-head attention runs **h parallel attention heads**, each with dimension d_k = d_model/h. Each head learns to attend to **different relationship types** (e.g., one head for syntactic dependencies, another for coreference, another for positional patterns). The outputs are concatenated and linearly projected: MultiHead(Q,K,V) = Concat(head_1,...,head_h)Wᴼ. This gives the model a richer representational capacity at the same computational cost as single-head attention.

---

### Q5: What are positional encodings and why are they needed?
**A:** Since self-attention is **permutation-equivariant** (it treats the input as a set, not a sequence), positional encodings inject order information. The original transformer uses sinusoidal functions: PE(pos,2i) = sin(pos/10000^(2i/d)), PE(pos,2i+1) = cos(pos/10000^(2i/d)). This allows the model to learn relative positions since PE(pos+k) can be expressed as a linear function of PE(pos). Modern alternatives include **learned positional embeddings** (BERT, GPT), **RoPE** (LLaMA — rotates Q,K vectors), and **ALiBi** (adds linear bias to attention scores).

---

### Q6: Encoder vs Decoder — what is the difference?
**A:** The **encoder** processes the full input with bidirectional self-attention (each token sees all others) and produces contextualized representations. The **decoder** generates output autoregressively with **masked (causal) self-attention** (each token only sees previous tokens) plus **cross-attention** to the encoder output. Key difference: the encoder builds understanding, the decoder generates output one token at a time. Encoder-only models (BERT) excel at understanding; decoder-only (GPT) at generation; encoder-decoder (T5) at sequence-to-sequence tasks.

---

### Q7: What is the masked attention in the decoder?
**A:** Masked (causal) self-attention prevents the decoder from attending to **future tokens** during training. It applies a triangular mask that sets future positions to -∞ before softmax, making those attention weights zero. This is essential because during inference the model generates one token at a time and cannot see the future, so training must simulate this constraint. Without masking, the model would "cheat" by looking ahead, and would fail at generation time.

---

### Q8: What is cross-attention?
**A:** Cross-attention (also called encoder-decoder attention) is where the decoder attends to the encoder output. The **queries come from the decoder**, while the **keys and values come from the encoder**. This is how the decoder accesses the source information (e.g., in translation, how the decoder "reads" the source sentence). It appears in the second sub-layer of each decoder block, after masked self-attention and before the feed-forward network.

---

### Q9: Why layer normalization instead of batch normalization?
**A:** Batch normalization normalizes across the batch dimension, which is problematic for sequences because: (1) **variable-length sequences** make batch statistics inconsistent, (2) batch stats are unreliable with **small batch sizes** common in NLP, and (3) batch norm creates dependency between samples. Layer normalization normalizes across the feature dimension **within each sample independently**, making it robust to variable lengths and small batches. It computes: LN(x) = γ · (x − μ) / √(σ² + ε) + β, where μ and σ are computed over the d_model dimension.

---

### Q10: What is the feed-forward network in transformers?
**A:** Each transformer block contains a **position-wise FFN** applied independently to each token: FFN(x) = max(0, xW₁ + b₁)W₂ + b₂. It has two linear layers with a ReLU (or GELU/SwiGLU in modern variants) activation in between. The inner dimension is typically **4× d_model** (e.g., 3072 for d_model=768). This is where the model stores **factual knowledge** ("memory" of the network) and performs non-linear feature transformation. It accounts for ~2/3 of the total parameters.

---

### Q11: How does the transformer handle variable-length inputs?
**A:** Transformers use **padding** to batch variable-length sequences together, adding special PAD tokens to shorter sequences. A **padding mask** is applied to attention scores (setting padded positions to -∞) so the model ignores padding tokens. For efficiency, modern frameworks use **packing** (concatenating multiple sequences into one with attention masks) or **dynamic batching** (grouping similar-length sequences). The self-attention mechanism itself is inherently flexible since it operates on sets of any size.

---

### Q12: What is teacher forcing?
**A:** Teacher forcing is a training strategy where the decoder receives the **ground-truth previous tokens** as input rather than its own predictions. This stabilizes and speeds up training by preventing error accumulation (exposure bias). However, it creates a **train-test mismatch** since at inference the model uses its own predictions. Mitigation strategies include: **scheduled sampling** (gradually using model predictions during training), **curriculum learning**, and modern approaches where decoder-only models naturally handle this via causal LM training.

---

### Q13: Pre-norm vs Post-norm — which is better?
**A:** **Post-norm** (original transformer): applies LayerNorm after the residual connection — x + LayerNorm(SubLayer(x)). **Pre-norm**: applies LayerNorm before the sublayer — x + SubLayer(LayerNorm(x)). Pre-norm is generally **better for training stability** because gradients flow more easily through the residual path without being modified by normalization. Pre-norm allows training **deeper models without warmup**, converges faster, but may have a slightly lower final performance ceiling. Most modern LLMs (GPT, LLaMA) use **pre-norm with RMSNorm** (simplified LayerNorm without mean centering).

---

### Q14: Why are residual connections important?
**A:** Residual connections (skip connections) add the input directly to the sublayer output: output = x + SubLayer(x). They solve the **vanishing gradient problem** in deep networks by providing a gradient highway — the gradient of the identity is 1, so gradients flow unimpeded through the skip path. They also enable **feature reuse** and make it easier for the network to learn identity mappings when needed. Without residual connections, training a 6+ layer transformer would be extremely difficult. They are used around **every sublayer** (attention and FFN).

---

### Q15: How do modern LLMs differ from the original transformer?
**A:** Key differences include:
- **Architecture**: Decoder-only (GPT, LLaMA) instead of encoder-decoder
- **Normalization**: Pre-norm with RMSNorm instead of post-norm LayerNorm
- **Positional encoding**: RoPE or ALiBi instead of sinusoidal
- **Activation**: SwiGLU/GeGLU instead of ReLU in FFN
- **Attention**: Grouped Query Attention (GQA) or Multi-Query Attention (MQA) for efficient KV caching
- **Scale**: Billions of parameters, trained on trillions of tokens
- **Training**: Next-token prediction + RLHF/DPO alignment
- **Context length**: Extended to 128K+ tokens via RoPE scaling, ring attention
- **Efficiency**: Flash Attention, KV caching, quantization (GPTQ, AWQ), speculative decoding

</div>

---

# 9. References & Citations

1. **Vaswani, A., et al.** (2017). *"Attention Is All You Need."* NeurIPS 2017. [arXiv:1706.03762](https://arxiv.org/abs/1706.03762)
2. **Devlin, J., et al.** (2019). *"BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding."* NAACL 2019. [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)
3. **Radford, A., et al.** (2018). *"Improving Language Understanding by Generative Pre-Training."* OpenAI. [GPT-1 Paper](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)
4. **Brown, T., et al.** (2020). *"Language Models are Few-Shot Learners."* NeurIPS 2020. [arXiv:2005.14165](https://arxiv.org/abs/2005.14165)
5. **Touvron, H., et al.** (2023). *"LLaMA: Open and Efficient Foundation Language Models."* Meta AI. [arXiv:2302.13971](https://arxiv.org/abs/2302.13971)
6. **Su, J., et al.** (2021). *"RoFormer: Enhanced Transformer with Rotary Position Embedding."* [arXiv:2104.09864](https://arxiv.org/abs/2104.09864)
7. **Dao, T., et al.** (2022). *"FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness."* NeurIPS 2022. [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)
8. **Shazeer, N.** (2019). *"Fast Transformer Decoding: One Write-Head is All You Need."* [arXiv:1911.02150](https://arxiv.org/abs/1911.02150)
9. **Xiong, R., et al.** (2020). *"On Layer Normalization in the Transformer Architecture."* ICML 2020. [arXiv:2002.04745](https://arxiv.org/abs/2002.04745)
10. **Zhang, B. & Sennrich, R.** (2019). *"Root Mean Square Layer Normalization."* NeurIPS 2019. [arXiv:1910.07467](https://arxiv.org/abs/1910.07467)
11. **Jay Alammar.** *"The Illustrated Transformer."* [jalammar.github.io](https://jalammar.github.io/illustrated-transformer/)
12. **Lilian Weng.** *"The Transformer Family (v2)."* [lilianweng.github.io](https://lilianweng.github.io/posts/2023-01-27-the-transformer-family-v2/)